# Introduction

Gu, Kelly, and Xiu study one of the central empirical asset-pricing problems: measuring conditional expected stock returns, or risk premiums. Their framing is deliberately predictive. For stock $i$ in month $t$, the object of interest is

$$
E_t(r_{i,t+1}) = g^*(z_{i,t}),
$$

where $r_{i,t+1}$ is the next-month excess return and $z_{i,t}$ is the vector of information known at time $t$. The paper asks which statistical method best approximates the unknown function $g^*(\cdot)$ when the predictor set is large, noisy, correlated, and potentially nonlinear.

The replication target is not a single trading rule. It build a large monthly stock panel, fit several model families under strict time-ordered validation, compare out-of-sample forecasts, and translate forecasts into economically interpretable portfolios. The original paper's main conclusion is that regularization is essential and that nonlinear interaction models, especially trees and neural networks, produce the strongest stock-level and portfolio-level forecasts.


Their main insight isa the expected excess return is the conditional expectation of a future realized excess return. This makes asset pricing a natural domain for machine learning, it is a difficult one because stock returns have low signal tonoise ratios, predictors are numerous and correlated, and overfitting can easily dominate apparent in-sample success.

The authors compare a broad model set: elastic net, principal components regression, partial least squares, generalized linear models with spline expansions and group lasso, random forests, gradient boosted regression trees, and feed-forward neural networks with one to five hidden layers.

Full OLS with all predictors performs poorly because it overfits. Penalized linear methods and dimension-reduction methods improve performance by controlling the effective number of parameters. Generalized linear models add univariate nonlinearities but do not dominate simpler regularized linear approaches. Trees and neural networks perform best because they can represent interactions among predictors. It converts forecasts into value-weighted decile portfolios and long-short spreads, then evaluates realized returns and Sharpe ratios. This portfolio analysis is the bridge from predictive accuracy to asset-pricing relevance.

The dominant predictors are price trends, liquidity, and volatility: short-term reversal, momentum, industry momentum, market equity, dollar volume, turnover, bid-ask spread, total volatility, idiosyncratic volatility.


HYPOTHESISS

1: Regularized models outperform unrestricted OLS out of sample. Full OLS should perform poorly when all baseline predictors are included, while OLS, elastic net, PCR, and PLS should improve out-of-sample prediction by reducing overfit.

2: Nonlinear interaction models outperform linear methods. Random forests, gradient boosted trees, and neural networks should improve over linear and generalized-linear models because they can capture interactions among stock characteristics and macro states.

3:  Forecast-sorted decile portfolios should show monotonicity in realized returns, and long-short portfolios should earn positive average excess returns and meaningful Sharpe ratios.

4: Important predictors are stable and economically interpretable. Variable importance rankings should emphasize momentum, liquidity, and volatility signals.


# Literature Review

Gu, Kelly, and Xiu (2020) sit at the intersection of empirical asset pricing and machine learning. The traditional cross-sectional literature studies expected returns as functions of a relatively small number of firm characteristics.

Fama and French (2008) providing central benchmarks for factor construction and characteristic-based return patterns. 
Lewellen (2015) is another important benchmark because it asks how well firm characteristics jointly explain the cross-section of expected returns.

The macro return-prediction side of the literature is represented by Welch and Goyal (2008), who organize and evaluate a standard set of aggregate predictors such as dividend-price ratios, earnings-price ratios, term spreads, default spreads, and stock variance. author import this macro-predictor tradition into the stock-level setting by interacting macro variables with firm characteristics.

The stock-characteristic universe follows Green, Hand, and Zhang (2017), who compile many return-predictive signals from the accounting and finance literature. it is constructing predictors with correct timing, lags, missing-value handling, and cross-sectional ranking.

The paper also builds on statistical learning references such as Hastie, Tibshirani, and Friedman (2009). Elastic net regularization, PCR, PLS, regression trees, random forests, gradient boosting, and neural networks are used not as black-box ornaments but as different ways to manage high-dimensional prediction and functional-form uncertainty.

Kozak, Nagel, and Santosh (2020) use shrinkage to control the high-dimensional cross-section, supporting the idea that asset-pricing signals should be treated jointly rather than anomaly by anomaly. 

Freyberger, Neuhierl, and Weber (2020) estimate nonlinear characteristic functions nonparametrically, supporting the paper's argument that linear specifications can miss important shape and interaction effects. 

Chen, Pelger, and Zhu (2019) impose no-arbitrage structure in a deep learning model, shifting the objective from pure return prediction toward economically disciplined pricing. 

Guijarro-Ordonez, Pelger, and Zanotti (2021) use deep learning for statistical arbitrage and emphasize residual portfolios and trading-policy design. 

Liao, Ma, Neuhierl, and Schilling (2025) focus on uncertainty around machine-learning return forecasts, which is a direct improvement over reporting point forecasts only. 

Chen and Gao (2024) study whether attribution methods in asset pricing satisfy risk-aware axioms, which is relevant for improving the interpretation of variable importance. 

Wang and Singh (2024) propose KAN-based autoencoders for factor models, suggesting a more interpretable nonlinear alternative to standard multilayer perceptrons. 

Ye, Goswami, Gu, Uddin, and Wang (2024) provide a broad review of how machine learning is reshaping empirical asset pricing, including explainability, overfitting, and alternative data. 

Zhang (2022) studies newer deep-learning architectures for risk-premium measurement and highlights distribution shift as a key challenge for financial time series.


# Supported Work and Methodology Improvements


Autoencoder-style and KAN-style factor models, including Wang and Singh (2024), suggest learning nonlinear factor exposures from characteristics rather than treating all forecasts as unconstrained black-box predictions. A natural improvement is to add an autoencoder benchmark that separates conditional betas, latent factors, and expected returns.

Liao, Ma, Neuhierl, and Schilling (2025) argue that ML forecasts in asset pricing should include uncertainty, not just point estimates. This improves the methodology by adding confidence intervals for expected returns, forecast dispersion by stock-month, and uncertainty-adjusted portfolio weights.

The original paper uses drop-in-R2 importance and derivative-based sensitivity. Chen and Gao (2024) motivate a stricter interpretability layer: compare drop-in-R2, permutation importance, SHAP/integrated-gradient-style attribution, and grouped risk-based importance to check whether explanations align with financial risk concepts.

Guijarro-Ordonez, Pelger, and Zanotti (2021) emphasize the distinction between predicting returns and forming tradable statistical-arbitrage strategies. The replication should therefore add transaction costs, turnover, capacity checks, leverage constraints, and residualized portfolio tests.

Zhang (2022) and broader review work such as Ye et al. (2024) emphasize that financial time series are nonstationary. Methodological improvements should include rolling performance diagnostics, regime-specific results, model decay analysis, and potentially recurrent, attention, or transformer baselines. These should be treated as extensions after the original feed-forward neural-network benchmark is replicated.



# Replication

## Data

The original paper uses monthly U.S. equity data from CRSP for NYSE, AMEX, and NASDAQ firms from March 1957 to December 2016. The risk-free rate is the Treasury-bill rate. Stock-level predictors consist of 94 firm characteristics based on Green, Hand, and Zhang (2017), plus 74 two-digit SIC industry dummies. Macro predictors follow Welch and Goyal (2008): dividend-price ratio (`dp`), earnings-price ratio (`ep`), book-to-market ratio (`bm`), net equity expansion (`ntis`), Treasury-bill rate (`tbl`), term spread (`tms`), default spread (`dfy`), and stock variance (`svar`).

The data has be organized as follows:

- `data/raw/crsp_monthly_returns.parquet`: monthly stock returns, identifiers, prices, shares, exchange/share codes, SIC, and market equity inputs.
- `data/raw/firm_characteristics.parquet`: stock characteristics with release-date-safe timestamps.
- `data/external/macro_predictors.parquet`: Welch-Goyal-style macro predictors.
- `data/external/risk_free_rate.parquet`: monthly Treasury-bill risk-free rate if not included with macro predictors.
- `data/processed/monthly_panel.parquet`: final one-row-per-stock-month modeling panel.

The final panel must include at minimum `date`, `permno`, `ret_excess_lead1`, and the feature columns. For economic portfolio tests it should also include `market_equity` and `sic2`.

Monthly characteristics should be available by the end of the prediction month, quarterly characteristics should be lagged at least four months, and annual characteristics should be lagged at least six months. Missing characteristic values should be handled cross-sectionally within each month, following the original paper's median-imputation approach before ranking.


## Data Processing Pipeline

The replication should process the data in a fixed order so that every predictor is known before the return it forecasts. The core unit of observation is one stock-month, indexed by `permno` and `date`.

### Input files

| File | Source | Required columns | Purpose |
|---|---|---|---|
| `data/raw/crsp_monthly_returns.parquet` | CRSP | `permno`, `date`, `ret`, `prc`, `shrout`, `siccd`, exchange/share-code fields if available | Monthly returns, market equity, universe filters, SIC industry code |
| `data/raw/firm_characteristics.parquet` | Green-Hand-Zhang-style characteristics from CRSP/Compustat | `permno`, `date`, 94 characteristic columns, update frequency or release-safe date | Main stock-level predictors |
| `data/external/macro_predictors.parquet` | Welch-Goyal data | `date`, `dp`, `ep`, `bm`, `ntis`, `tbl`, `tms`, `dfy`, `svar` | Macro state vector `x_t` |
| `data/external/risk_free_rate.parquet` | Treasury-bill/risk-free source | `date`, `rf` | Excess-return target construction |
| `data/processed/monthly_panel.parquet` | Produced by this project | `date`, `permno`, `ret_excess_lead1`, features, `market_equity`, `sic2` | Final model-ready panel |

### Step 1: Standardize dates and identifiers

Convert all dates to month-end timestamps and keep one row per `permno`-month. The core key is

$$
key_{i,t} = (permno_i, date_t).
$$

Returns, characteristics, macro predictors, and risk-free rates must all use the same monthly calendar.

### Step 2: Build market equity and industry code

Use CRSP price and shares outstanding to construct market equity:

$$
ME_{i,t} = |PRC_{i,t}| \times SHROUT_{i,t}.
$$

Construct two-digit SIC industry code:

$$
sic2_{i,t}=\left\lfloor \frac{SIC_{i,t}}{100} \right\rfloor.
$$

Then one-hot encode `sic2` for the 74 industry dummies used by the paper {cite}`GuKellyXiu2020`.

### Step 3: Construct lead excess return

Merge the risk-free rate by month and compute current excess return:

$$
r^e_{i,t}=r_{i,t}-r^f_t.
$$

The prediction target is next-month excess return:

$$
ret\_excess\_lead1_{i,t}=r^e_{i,t+1}.
$$

In code this is a `groupby("permno").shift(-1)` after sorting by `permno` and `date`.

### Step 4: Apply release lags to characteristics

To avoid look-ahead bias, characteristics must be available by the prediction date. The paper assumes approximately:

- Monthly characteristics: available by month `t`.
- Quarterly accounting variables: use the most recent value available by `t - 4` months.
- Annual accounting variables: use the most recent value available by `t - 6` months.

For every characteristic $c_j$, the modeling value should satisfy

$$
release\_date(c_{i,j}) \leq date_t.
$$

### Step 5: Merge stock returns, characteristics, macro variables, and industry dummies

The final preprocessed row is

$$
\left[date_t, permno_i, ret\_excess\_lead1_{i,t}, ME_{i,t}, sic2_{i,t}, c_{i,t}, x_t\right].
$$

Macro variables are common across stocks in month `t`, so they merge by `date`. Characteristics merge by `permno` and `date`.

### Step 6: Impute missing characteristics cross-sectionally

For each month and characteristic, fill missing values with the cross-sectional median:

$$
c_{i,t,j}^{filled}=\begin{cases}
c_{i,t,j}, & c_{i,t,j}\text{ observed},\\
median_i(c_{i,t,j}), & c_{i,t,j}\text{ missing}.
\end{cases}
$$

Do this before ranking so missing values become neutral within each month.

### Step 7: Rank characteristics to `[-1, 1]`

For each month and characteristic, compute percentile rank and map it to the paper's scale:

$$
ranked\_c_{i,t,j}=-1+2\times percentile\_rank(c_{i,t,j}).
$$

This makes predictors comparable across time and reduces the effect of extreme raw accounting values.

### Step 8: Create macro-characteristic interactions

For each ranked characteristic and macro predictor, create

$$
z_{i,t,j,k}=ranked\_c_{i,t,j}\times x_{t,k}.
$$

The full baseline feature vector is

$$
z_{i,t}=x_t\otimes c_{i,t},
$$

plus industry dummies.




## Hypothesis Tests

The core forecast test is panel out-of-sample $R^2$ against a zero benchmark:

$$
R^2_{oos} = 1 - \frac{\sum_{(i,t) \in T_3} (r_{i,t+1} - \hat r_{i,t+1})^2}{\sum_{(i,t) \in T_3} r_{i,t+1}^2}.
$$

A positive value means the model beats the naive forecast that every stock's future excess return is zero. This benchmark is intentionally stricter than historical-mean forecasts for individual stocks.

Pairwise model comparisons should use a Diebold-Mariano-style statistic on cross-sectional average squared-error differences. For models A and B, compute each month:

$$
d_{AB,t+1} = \frac{1}{n_t}\sum_i \left(e^2_{A,i,t+1} - e^2_{B,i,t+1}\right).
$$

Then test whether the time-series mean of `d` differs from zero, using Newey-West standard errors. A positive value means model B has lower squared error than model A under this sign convention.

The economic tests sort stocks into deciles by forecast each month, compute value-weighted decile returns, and report the high-minus-low spread. The main outputs are average monthly return, annualized Sharpe ratio, monotonicity across deciles, and performance for large-stock and small-stock subsamples.


## Replication of Key Analytical Techniques

This section states the mathematical specification that the replication should implement. The notation follows Gu, Kelly, and Xiu (2020) unless otherwise noted {cite}`GuKellyXiu2020`.

### 1. Target and information set

For stock $i$ and month $t$, the excess-return model is

$$
r_{i,t+1} = E_t(r_{i,t+1}) + \epsilon_{i,t+1},
$$

with conditional expectation

$$
E_t(r_{i,t+1}) = g^*(z_{i,t}).
$$

The replication target is therefore to estimate a forecasting rule

$$
\hat r_{i,t+1} = \hat g(z_{i,t})
$$

using only information observable by the end of month $t$.

### 2. Feature construction

Let $c_{i,t}$ be the vector of stock characteristics and $x_t$ be the vector of macro predictors plus a constant. The paper's baseline stock-level covariates are

$$
z_{i,t} = x_t \otimes c_{i,t},
$$

where $\otimes$ is the Kronecker product. With 94 characteristics, 8 macro predictors, one constant, and 74 industry dummies, the baseline feature count is

$$
94 \times (8 + 1) + 74 = 920.
$$

Each characteristic is ranked cross-sectionally each month and mapped to `[-1, 1]`. If $rank_{i,t,j}$ is the percentile rank of characteristic $j$, a simple implementation is

$$
\tilde c_{i,t,j} = -1 + 2 \times rank_{i,t,j}.
$$

### 3. Pooled OLS

The simple linear model is

$$
g(z_{i,t};\theta) = z_{i,t}'\theta.
$$

The pooled least-squares objective is

$$
\hat\theta_{OLS} = \arg\min_{\theta}\frac{1}{NT}\sum_{i=1}^{N}\sum_{t=1}^{T}\left(r_{i,t+1} - z_{i,t}'\theta\right)^2.
$$

This model is included as a benchmark. In the original paper, unrestricted OLS with the full 920-feature set performs poorly because the predictor set is too large and noisy relative to the signal {cite}`GuKellyXiu2020`.

### 4. OLS-3 benchmark

The sparse OLS benchmark uses only three characteristics: size, book-to-market/value, and momentum. Let

$$
z^{(3)}_{i,t} = \left[ size_{i,t}, value_{i,t}, momentum_{i,t} \right].
$$

Then

$$
\hat r_{i,t+1}^{OLS3} = (z^{(3)}_{i,t})'\hat\theta_3.
$$

This benchmark checks whether machine learning improves over a deliberately simple characteristic model.

### 5. Weighted and Huber robust loss

The paper also discusses weighted least squares:

$$
L_W(\theta)=\frac{1}{NT}\sum_{i=1}^{N}\sum_{t=1}^{T}w_{i,t}\left(r_{i,t+1}-g(z_{i,t};\theta)\right)^2.
$$

For heavy-tailed financial returns, Huber loss replaces squared loss with

$$
H(u;\xi)=
\begin{cases}
u^2, & |u| \leq \xi, \\
2\xi |u| - \xi^2, & |u| > \xi,
\end{cases}
$$

where

$$
u = r_{i,t+1} - g(z_{i,t};\theta).
$$

The robust objective is

$$
L_H(\theta)=\frac{1}{NT}\sum_{i=1}^{N}\sum_{t=1}^{T}H\left(r_{i,t+1}-g(z_{i,t};\theta);\xi\right).
$$

### 6. Elastic net

Elastic net keeps the linear forecast form but regularizes coefficients:

$$
\hat\theta_{EN}=\arg\min_{\theta}\left[L(\theta)+\phi(\theta;\lambda,\rho)\right],
$$

where

$$
\phi(\theta;\lambda,\rho)=\lambda(1-\rho)\sum_{j=1}^{P}|\theta_j| + \frac{\lambda\rho}{2}\sum_{j=1}^{P}\theta_j^2.
$$

The $L_1$ part performs variable selection and the $L_2$ part shrinks correlated predictors. The tuning parameters $\lambda$ and $\rho$ are chosen on the validation sample.

### 7. Principal components regression (PCR)

Stack all stock-month observations into a target vector $R$ and predictor matrix $Z$:

$$
R = Z\theta + E.
$$

PCR replaces $Z$ with $K$ principal components:

$$
R = (Z\Omega_K)\theta_K + \tilde E.
$$

The $j$th principal-component weight vector solves

$$
w_j = \arg\max_w Var(Zw),
$$

subject to

$$
w'w=1, \quad Cov(Zw, Zw_l)=0 \quad \text{for } l<j.
$$

PCR chooses components that preserve predictor covariance, then estimates $\theta_K$ by OLS. The number of components $K$ is validation-tuned.

### 8. Partial least squares (PLS)

PLS also uses

$$
R = (Z\Omega_K)\theta_K + \tilde E,
$$

but chooses components to maximize association with future returns. The $j$th PLS component solves

$$
w_j = \arg\max_w Cov^2(R, Zw),
$$

subject to

$$
w'w=1, \quad Cov(Zw, Zw_l)=0 \quad \text{for } l<j.
$$

This makes PLS more supervised than PCR because return predictability enters the dimension-reduction step {cite}`KellyPruitt2013 KellyPruitt2015`.

### 9. Generalized linear model with spline expansion

The generalized linear model adds nonlinear transformations of each predictor but remains additive across predictors:

$$
g(z;\theta,p)=\sum_{j=1}^{P}p(z_j)'\theta_j.
$$

The paper uses second-order spline basis functions such as

$$
p(z_j)=\left[1, z_j, (z_j-c_1)^2, \ldots, (z_j-c_{K-2})^2\right]'.
$$

To control the enlarged parameter set, the model uses group lasso:

$$
\phi(\theta;\lambda,K)=\lambda\sum_{j=1}^{P}\left(\sum_{k=1}^{K}\theta_{j,k}^2\right)^{1/2}.
$$

This selects or removes all spline terms for a predictor as a group.

### 10. Regression tree

A tree partitions the predictor space into $K$ terminal leaves $C_k(L)$ with maximum depth $L$. The prediction rule is

$$
g(z_{i,t};\theta,K,L)=\sum_{k=1}^{K}\theta_k \mathbf{1}\{z_{i,t}\in C_k(L)\}.
$$

For a leaf $C_k$, the fitted value is the average outcome inside the leaf:

$$
\theta_k = \frac{1}{|C_k|}\sum_{(i,t):z_{i,t}\in C_k}r_{i,t+1}.
$$

A split is chosen to reduce squared-error impurity:

$$
H(\theta,C)=\frac{1}{|C|}\sum_{z_{i,t}\in C}\left(r_{i,t+1}-\theta\right)^2.
$$

A tree of depth $L$ can represent interactions up to roughly order $L-1$, which is why tree models are important for this paper's interaction hypothesis.

### 11. Random forest

A random forest averages predictions from many bootstrapped trees:

$$
\hat g_{RF}(z)=\frac{1}{B}\sum_{b=1}^{B}T_b(z;\hat\theta_b).
$$

Each tree is trained on a bootstrap sample, and each split considers only a random subset of predictors. The main validation-tuned hyperparameters are the number of trees $B$, tree depth $L$, minimum leaf size, and number of candidate split variables.

### 12. Gradient boosted regression trees

Gradient boosting builds an additive ensemble of shallow trees. Initialize with a constant forecast $F_0(z)$. At step $b$, fit a tree $T_b(z)$ to the current residuals or negative gradient, then update

$$
F_b(z)=F_{b-1}(z)+\nu T_b(z),
$$

where $\nu\in(0,1)$ is the learning rate. The final forecast is

$$
\hat g_{GBRT}(z)=F_B(z)=F_0(z)+\nu\sum_{b=1}^{B}T_b(z).
$$

The main tuning parameters are tree depth $L$, learning rate $\nu$, and number of boosting iterations $B$.

### 13. Feed-forward neural networks

The neural-network input layer is

$$
x^{(0)} = [1,z_1,\ldots,z_P]'.
$$

For hidden layer $l$, neuron $k$ is computed recursively as

$$
x_k^{(l)} = ReLU\left((x^{(l-1)})'\theta_k^{(l-1)}\right),
$$

with

$$
ReLU(a)=\max(0,a).
$$

The final output is

$$
g(z;\theta)= (x^{(L-1)})'\theta^{(L-1)}.
$$

The paper studies five architectures:

$$
NN1=[32],\quad NN2=[32,16],\quad NN3=[32,16,8],
$$

$$
NN4=[32,16,8,4],\quad NN5=[32,16,8,4,2].
$$

Training minimizes penalized prediction loss and uses regularization devices such as early stopping, learning-rate shrinkage, batch normalization, and ensembling across random seeds {cite}`GuKellyXiu2020 ChenPelgerZhu2019`.

### 14. Validation and testing protocol

For each rolling split, the workflow is

$$
\theta(\eta) = \arg\min_{\theta}L_{train}(\theta;\eta),
$$

where $\eta$ denotes hyperparameters. Choose

$$
\hat\eta = \arg\min_{\eta}L_{validation}(\theta(\eta);\eta).
$$

Then evaluate only on the test set:

$$
\hat r_{i,t+1}=g(z_{i,t};\hat\theta(\hat\eta)), \quad (i,t)\in T_{test}.
$$

The original design begins with 18 training years, 12 validation years, and then annual rolling or expanding test evaluation.

### 15. Forecast accuracy

The main panel forecast metric is out-of-sample $R^2$ against a zero forecast:

$$
R^2_{oos}=1-\frac{\sum_{(i,t)\in T_3}(r_{i,t+1}-\hat r_{i,t+1})^2}{\sum_{(i,t)\in T_3}r_{i,t+1}^2}.
$$

The denominator is not demeaned because the benchmark is the zero excess-return forecast, not the historical mean.

### 16. Pairwise model comparison

For two models A and B, define the cross-sectional average squared-error difference each month:

$$
d_{AB,t+1}=\frac{1}{n_t}\sum_i\left(e^2_{A,i,t+1}-e^2_{B,i,t+1}\right),
$$

where

$$
e_{A,i,t+1}=r_{i,t+1}-\hat r^A_{i,t+1}.
$$

The Diebold-Mariano-style statistic is

$$
DM_{AB}=\frac{\bar d_{AB}}{\widehat{se}(\bar d_{AB})},
$$

with Newey-West standard errors for $\bar d_{AB}$ {cite}`DieboldMariano1995 GuKellyXiu2020`.

### 17. Variable importance

The paper's first importance measure is the reduction in out-of-sample $R^2$ after setting predictor $j$ to zero:

$$
VI_j = R^2_{oos} - R^2_{oos,-j}.
$$

For differentiable models, a derivative-based sensitivity measure is

$$
SSD_j = \sum_{(i,t)\in T}\left(\frac{\partial g(z;\theta)}{\partial z_j}\bigg|_{z=z_{i,t}}\right)^2.
$$

Tree models can instead report mean decrease in impurity or permutation importance.

### 18. Portfolio sorts

Each month, sort stocks into deciles by forecast $\hat r_{i,t+1}$. For decile $q$, the value-weighted return is

$$
R_{q,t+1}=\sum_{i\in q_t}w_{i,t}r_{i,t+1},
$$

where

$$
w_{i,t}=\frac{ME_{i,t}}{\sum_{j\in q_t}ME_{j,t}}.
$$

The long-short spread is

$$
R^{LS}_{t+1}=R_{10,t+1}-R_{1,t+1}.
$$

Annualized Sharpe ratio is

$$
SR_{ann}=\sqrt{12}\frac{\bar R^{LS}}{\sigma(R^{LS})}.
$$

These portfolio equations turn the statistical forecast into the economic test of whether the model produces useful expected-return rankings.


## Analysis Process and Expected Outputs

After the model-ready panel exists, the analysis should run in the following order.

### Analysis 1: Descriptive data checks

Produce a table with:

- number of stock-month observations,
- number of unique stocks,
- sample start and end dates,
- average stocks per month,
- missing-value rate by characteristic family,
- distribution of `ret_excess_lead1`,
- distribution of `market_equity`,
- number of industries each month.

These checks verify that the replication sample resembles the original paper's broad CRSP universe.

### Analysis 2: Rolling train-validation-test splits

Generate time-ordered splits. The starting design is 18 training years, 12 validation years, and 1 test year per refit:

$$
T_1 = \text{training},\quad T_2 = \text{validation},\quad T_3 = \text{testing}.
$$

For each split, estimate model parameters on $T_1$, tune hyperparameters on $T_2$, and store forecasts for $T_3$ only.

### Analysis 3: Model-level forecast tables

For each model, store a prediction panel with columns:

```text
date, permno, ret_excess_lead1, forecast, model, market_equity, sic2
```

Then compute:

$$
R^2_{oos}=1-\frac{\sum(r-\hat r)^2}{\sum r^2}.
$$

Report the main table by model: `OLS`, `OLS-3`, `ENet`, `PCR`, `PLS`, `GLM`, `RF`, `GBRT`, `NN1`-`NN5`.

### Analysis 4: Pairwise model comparison

For each model pair A and B, compute monthly cross-sectional average error differences:

$$
d_{AB,t+1}=\frac{1}{n_t}\sum_i(e^2_{A,i,t+1}-e^2_{B,i,t+1}).
$$

Report a Diebold-Mariano-style statistic:

$$
DM_{AB}=\frac{\bar d_{AB}}{\widehat{se}(\bar d_{AB})}.
$$

This produces a matrix like the paper's pairwise model comparison table.

### Analysis 5: Portfolio sorts

For every month and model:

1. Sort stocks into deciles by forecast.
2. Compute value-weighted return for each decile.
3. Compute high-minus-low spread:

$$
R^{LS}_{t+1}=R_{10,t+1}-R_{1,t+1}.
$$

4. Report mean monthly return, annualized volatility, annualized Sharpe, and monotonicity across deciles.

### Analysis 6: Subsample analysis

Repeat forecast and portfolio analysis for:

- top 1,000 stocks by market equity,
- bottom 1,000 stocks by market equity,
- pre-2000 vs post-2000 periods,
- high-volatility vs low-volatility macro regimes,
- with and without financial firms if the required identifiers are available.

### Analysis 7: Variable importance

For each trained model and feature group, report importance using:

$$
VI_j = R^2_{oos} - R^2_{oos,-j}.
$$

Group importance should also be computed for:

- momentum and reversal,
- liquidity,
- volatility and beta,
- valuation,
- profitability and investment,
- macro interactions,
- industry dummies.


